# PyTorch for Finance

PyTorch is a general purpose tensor computation and automatic differentiation library, originally built for deep learning, but useful across quantitative finance far beyond neural networks. Its automatic differentiation engine can compute exact option Greeks without deriving a single closed-form formula by hand, its GPU-accelerated tensor operations can price and simulate at a scale that pure NumPy struggles with, and its optimizer machinery can solve constrained portfolio and calibration problems by gradient descent instead of specialized quadratic programming solvers.

**A note on how this notebook was built.** PyTorch could not be installed in the sandboxed environment used to author this notebook (its network access is restricted to a small package whitelist that does not include a PyTorch wheel). Every PyTorch code cell below is written to be correct, complete, and runnable as-is once you execute it in an environment with `pip install torch` available, but it has not been executed here. To keep this notebook trustworthy despite that, every finance application below pairs its PyTorch cell with a NumPy or SciPy cell computing the same answer by an independent, closed-form or well-established numerical method, and that companion cell **has** been executed, so you have a genuine target value to check your PyTorch output against the moment you run it.

**What this notebook covers:**

Part 1, PyTorch fundamentals: tensors, automatic differentiation, building a model with `nn.Module`, loss functions and optimizers, and the `Dataset`/`DataLoader` pattern for batching.

Part 2, finance applications: computing option Greeks by automatic differentiation, a neural network for cross-sectional return prediction, a neural network as a fast option pricing surrogate, portfolio optimization by gradient descent, and an LSTM for volatility forecasting.

## Part 1: PyTorch Fundamentals

### 1.1 Tensors

A tensor is PyTorch's core data structure, an n-dimensional array much like a NumPy array, but with two extra capabilities that matter for everything that follows: it can track the operations performed on it for automatic differentiation, and it can live on a GPU for fast parallel computation. Tensors support the same broadcasting, slicing, and reshaping rules NumPy users already know.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)

# creation
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.zeros((3, 4))
c = torch.arange(0, 10, dtype=torch.float32).reshape(2, 5)

print("a:", a)
print("b shape:", b.shape)
print("c:\n", c)

# basic operations, broadcasting exactly like NumPy
prices = torch.tensor([100.0, 102.5, 98.3, 105.1])
returns = (prices[1:] - prices[:-1]) / prices[:-1]
print("simple returns:", returns)

# device placement: this line moves a tensor to GPU if one is available, CPU otherwise
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
prices_on_device = prices.to(device)
print("device in use:", device)

### 1.2 Automatic Differentiation

Setting `requires_grad=True` on a tensor tells PyTorch to record every operation applied to it, building a computation graph. Calling `.backward()` on a scalar output then walks that graph backward and fills in `.grad` on every leaf tensor with the exact derivative of the output with respect to that tensor, computed via the chain rule, not a finite-difference approximation. This single mechanism is what later lets Greeks be computed automatically instead of derived by hand.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2*x + 1   # y = (x+1)^2, dy/dx = 2x + 2

y.backward()
print("x =", x.item(), " y =", y.item(), " dy/dx =", x.grad.item(), " (expected 2*3+2=8)")

# a vector-valued example: gradient of a portfolio's variance with respect to its weights
w = torch.tensor([0.4, 0.35, 0.25], requires_grad=True)
cov = torch.tensor([[0.04, 0.01, 0.02],
                     [0.01, 0.09, 0.015],
                     [0.02, 0.015, 0.0625]])
port_var = w @ cov @ w
port_var.backward()
print("portfolio variance:", port_var.item())
print("gradient of variance w.r.t. weights:", w.grad)

### 1.3 Building a Model with `nn.Module`

`nn.Module` is the base class for every PyTorch model, from a single linear layer to a full transformer. Subclassing it requires defining the layers in `__init__` and the computation in `forward`; PyTorch automatically tracks every parameter inside the layers you assign as attributes, so they are all discovered for optimization without any manual bookkeeping.

In [ ]:
class SimpleReturnPredictor(nn.Module):
    def __init__(self, n_features, hidden_size=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

model = SimpleReturnPredictor(n_features=3)
print(model)

n_params = sum(p.numel() for p in model.parameters())
print("total trainable parameters:", n_params)

### 1.4 Loss Functions and Optimizers

A loss function measures how far the model's predictions are from the target; `nn.MSELoss` is the standard choice for a regression problem like return prediction. An optimizer then uses the gradients computed by `.backward()` to update every parameter in the direction that reduces the loss. Adam, an adaptive variant of stochastic gradient descent, is the most common default choice for training neural networks in practice.

In [ ]:
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# one manual training step, to show the mechanics explicitly before the full training loop below
x_batch = torch.randn(8, 3)
y_batch = torch.randn(8)

optimizer.zero_grad()          # clear gradients from any previous step
pred = model(x_batch)          # forward pass
loss = loss_fn(pred, y_batch)  # compute the loss
loss.backward()                # backward pass, fills in .grad on every parameter
optimizer.step()                # update every parameter using its gradient

print("loss on this batch:", loss.item())

### 1.5 Datasets and DataLoaders

For anything beyond a toy example, data needs to be batched, shuffled, and fed to the model efficiently. Wrapping data in a `Dataset` and passing it to a `DataLoader` handles all of this, including, when needed, loading data lazily from disk rather than holding everything in memory at once.

In [ ]:
class ReturnDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

import numpy as np
X_demo = np.random.randn(100, 3)
y_demo = np.random.randn(100)
dataset = ReturnDataset(X_demo, y_demo)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

first_batch_X, first_batch_y = next(iter(loader))
print("one batch of features has shape:", first_batch_X.shape)
print("number of batches per epoch:", len(loader))

## Part 2: Finance Applications

### 2.1 Option Greeks by Automatic Differentiation

The Black-Scholes Greeks (delta, gamma, vega) are ordinarily derived by hand, differentiating the pricing formula term by term, a genuinely tedious exercise for anything more complex than a plain vanilla option, and one that has to be redone from scratch for every new payoff. Automatic differentiation removes that step entirely: write the pricing formula once as ordinary PyTorch tensor operations, and `.backward()` computes every Greek exactly, no matter how the payoff is defined.

First, the target values, computed the traditional way from the closed-form Black-Scholes formulas, so there is a known answer to check the PyTorch version against.

In [1]:
import numpy as np
from scipy.stats import norm

def bs_price(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

def bs_greeks_closed_form(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    delta = norm.cdf(d1)
    gamma = norm.pdf(d1) / (S*sigma*np.sqrt(T))
    vega = S*norm.pdf(d1)*np.sqrt(T)
    return delta, gamma, vega

S, K, T, r, sigma = 100.0, 100.0, 0.5, 0.03, 0.22
price = bs_price(S, K, T, r, sigma)
delta, gamma, vega = bs_greeks_closed_form(S, K, T, r, sigma)

print("TARGET VALUES (closed form, computed with SciPy):")
print(f"  price = {price:.6f}")
print(f"  delta = {delta:.6f}")
print(f"  gamma = {gamma:.6f}")
print(f"  vega  = {vega:.6f}")

TARGET VALUES (closed form, computed with SciPy):
  price = 6.926611
  delta = 0.569148
  gamma = 0.025259
  vega  = 27.784666


Now the same price, written as a plain PyTorch computation, with the Greeks recovered from a single `.backward()` call. `create_graph=True` on the first backward pass is what allows a second derivative, gamma, to be taken afterward.

In [ ]:
def bs_price_torch(S, K, T, r, sigma):
    # the standard normal CDF, expressed with the error function so the whole computation
    # stays inside PyTorch's autograd-tracked operations
    N = lambda x: 0.5 * (1 + torch.erf(x / torch.sqrt(torch.tensor(2.0))))
    d1 = (torch.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*torch.sqrt(T))
    d2 = d1 - sigma*torch.sqrt(T)
    return S*N(d1) - K*torch.exp(-r*T)*N(d2)

S_t = torch.tensor(100.0, requires_grad=True)
sigma_t = torch.tensor(0.22, requires_grad=True)
K_t = torch.tensor(100.0)
T_t = torch.tensor(0.5)
r_t = torch.tensor(0.03)

price_t = bs_price_torch(S_t, K_t, T_t, r_t, sigma_t)
delta_t, vega_t = torch.autograd.grad(price_t, [S_t, sigma_t], create_graph=True)
gamma_t = torch.autograd.grad(delta_t, S_t, retain_graph=True)[0]

print("AUTOGRAD VALUES (run this cell to fill these in):")
print("price:", price_t.item())
print("delta:", delta_t.item())
print("gamma:", gamma_t.item())
print("vega: ", vega_t.item())
print("these should match the target values above to within numerical precision")

### 2.2 A Neural Network for Cross-Sectional Return Prediction

Linear factor models (the momentum, value, and quality signals from earlier notebooks in this series) assume a stock's expected return is a linear combination of its characteristics. A neural network relaxes that assumption, letting the model discover nonlinear interactions between characteristics on its own, at the cost of needing more data and more careful tuning to avoid overfitting.

The synthetic data below has a genuine, modest nonlinear interaction built in (between momentum and quality, and a diminishing-returns effect in value), on top of the same linear exposures a factor model would already capture, so a neural network has real, but not unlimited, room to add value. A linear regression baseline is fit first as the target to beat.

In [2]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr

rng = np.random.default_rng(42)

n = 4000
momentum = rng.normal(0, 1, n)
value = rng.normal(0, 1, n)
quality = rng.normal(0, 1, n)

true_signal = 0.35*momentum + 0.25*value + 0.20*quality + 0.15*momentum*quality - 0.10*value**2
noise = rng.normal(0, 1.6, n)
forward_return = true_signal + noise

X = np.column_stack([momentum, value, quality])
y = forward_return
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

lr = LinearRegression().fit(X_train, y_train)
pred_lr = lr.predict(X_test)
ic_lr, _ = spearmanr(pred_lr, y_test)
mse_lr = np.mean((pred_lr - y_test) ** 2)

print("TARGET TO BEAT, linear regression baseline:")
print(f"  information coefficient (IC) = {ic_lr:.4f}")
print(f"  mean squared error (MSE)     = {mse_lr:.4f}")
print("a neural network that captures the true momentum x quality interaction and the value")
print("curvature should improve modestly on both numbers without dramatically outperforming,")
print("since roughly half of the true signal is already linear and the noise floor is high")

TARGET TO BEAT, linear regression baseline:
  information coefficient (IC) = 0.2426
  mean squared error (MSE)     = 2.5789
a neural network that captures the true momentum x quality interaction and the value
curvature should improve modestly on both numbers without dramatically outperforming,
since roughly half of the true signal is already linear and the noise floor is high


The PyTorch version: the same three factors, the `SimpleReturnPredictor` model from Part 1, trained with early stopping on a held-out validation split to avoid overfitting the nonlinear capacity onto noise.

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

# carve out a validation split from the training data for early stopping
n_val = 400
X_fit, y_fit = X_train_t[:-n_val], y_train_t[:-n_val]
X_val, y_val = X_train_t[-n_val:], y_train_t[-n_val:]

model = SimpleReturnPredictor(n_features=3, hidden_size=16)
optimizer = optim.Adam(model.parameters(), lr=5e-3, weight_decay=1e-4)
loss_fn = nn.MSELoss()

best_val_loss = float('inf')
best_state = None
patience, patience_counter = 15, 0

for epoch in range(300):
    model.train()
    optimizer.zero_grad()
    pred = model(X_fit)
    loss = loss_fn(pred, y_fit)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(X_val), y_val).item()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"early stopping at epoch {epoch}")
            break

model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    pred_nn = model(X_test_t).numpy()

from scipy.stats import spearmanr as sr
ic_nn, _ = sr(pred_nn, y_test)
mse_nn = np.mean((pred_nn - y_test) ** 2)
print(f"neural network IC  = {ic_nn:.4f}  (linear baseline was {ic_lr:.4f})")
print(f"neural network MSE = {mse_nn:.4f}  (linear baseline was {mse_lr:.4f})")

### 2.3 A Neural Network as an Option Pricing Surrogate

Pricing options in bulk, across thousands of strikes, maturities, and volatility scenarios for a risk report or a real-time trading system, means calling the pricing formula an enormous number of times. Even a fast closed-form formula like Black-Scholes adds up at scale, and for models without a closed form (many stochastic volatility or American-style models), the per-call cost is far higher, often requiring their own Monte Carlo simulation or finite-difference solve. A common practical solution is a surrogate model: train a neural network once, offline, to learn the mapping from an option's inputs to its price, then use the trained network for fast, repeated inference afterward. The training data below comes from Black-Scholes purely because it has a known correct answer to validate the surrogate against; the same approach works for pricing models with no closed form at all.

In [3]:
import numpy as np
from scipy.stats import norm

def bs_price(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

rng = np.random.default_rng(3)
n_samples = 20000
moneyness = rng.uniform(0.7, 1.3, n_samples)   # S / K
T_samples = rng.uniform(0.05, 2.0, n_samples)
sigma_samples = rng.uniform(0.10, 0.50, n_samples)
r_fixed = 0.03

K_fixed = 100.0
S_samples = moneyness * K_fixed
price_samples = bs_price(S_samples, K_fixed, T_samples, r_fixed, sigma_samples)
norm_price = price_samples / K_fixed   # price normalized by strike, a standard surrogate-model input transform

# a held-out test grid the surrogate has never seen, spanning the same input ranges
test_moneyness = np.array([0.8, 0.9, 1.0, 1.1, 1.2])
test_T = np.full(5, 0.5)
test_sigma = np.full(5, 0.25)
test_S = test_moneyness * K_fixed
test_price_true = bs_price(test_S, K_fixed, test_T, r_fixed, test_sigma) / K_fixed

print("TARGET VALUES on a held-out test grid (T=0.5, sigma=0.25, K=100, varying moneyness):")
for m, p in zip(test_moneyness, test_price_true):
    print(f"  moneyness={m:.2f}: normalized price = {p:.5f}")

TARGET VALUES on a held-out test grid (T=0.5, sigma=0.25, K=100, varying moneyness):
  moneyness=0.80: normalized price = 0.00920
  moneyness=0.90: normalized price = 0.03229
  moneyness=1.00: normalized price = 0.07760
  moneyness=1.10: normalized price = 0.14467
  moneyness=1.20: normalized price = 0.22763


The surrogate network: three inputs (moneyness, time to maturity, volatility), a small feedforward network, trained on the sampled grid above and evaluated against the held-out test points.

In [ ]:
class PricingSurrogate(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1),
            nn.Softplus()   # a price is always non-negative, Softplus enforces that structurally
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

X_sur = np.column_stack([moneyness, T_samples, sigma_samples])
X_sur_t = torch.tensor(X_sur, dtype=torch.float32)
y_sur_t = torch.tensor(norm_price, dtype=torch.float32)

surrogate = PricingSurrogate(hidden_size=32)
optimizer = optim.Adam(surrogate.parameters(), lr=3e-3)
loss_fn = nn.MSELoss()

dataset = ReturnDataset(X_sur, norm_price)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

for epoch in range(60):
    epoch_loss = 0.0
    for xb, yb in loader:
        optimizer.zero_grad()
        pred = surrogate(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(yb)
    if epoch % 10 == 0:
        print(f"epoch {epoch}: mean squared error = {epoch_loss/len(dataset):.7f}")

surrogate.eval()
X_test_grid = np.column_stack([test_moneyness, test_T, test_sigma])
with torch.no_grad():
    surrogate_pred = surrogate(torch.tensor(X_test_grid, dtype=torch.float32)).numpy()

print("\nSURROGATE PREDICTIONS vs. TARGET (should be close after training converges):")
for m, p_sur, p_true in zip(test_moneyness, surrogate_pred, test_price_true):
    print(f"  moneyness={m:.2f}: surrogate={p_sur:.5f}  target={p_true:.5f}  error={abs(p_sur-p_true):.5f}")

### 2.4 Portfolio Optimization by Gradient Descent

Mean-variance portfolio optimization is normally solved with a quadratic programming solver or, in the unconstrained case, a closed-form matrix expression. Framing it instead as a gradient descent problem is a useful exercise for a different reason: it shows how naturally PyTorch's optimizer machinery extends beyond neural networks to any problem with a differentiable objective, including ones with no learning or prediction involved at all.

Long-only, fully-invested weights (no shorting, weights sum to one) are enforced by parametrizing the weights as a softmax of unconstrained scores, exactly the same trick used to turn raw network outputs into a valid probability distribution in a classification model. The target below solves the identical constrained problem with SciPy's general-purpose SLSQP solver, an independent method to check the gradient descent result against.

In [4]:
import numpy as np
from scipy.optimize import minimize

rng = np.random.default_rng(7)

n_assets = 6
true_mean = np.array([0.09, 0.07, 0.11, 0.05, 0.08, 0.06]) / 252
vols = np.array([0.18, 0.14, 0.25, 0.10, 0.20, 0.12]) / np.sqrt(252)
corr = np.array([
    [1.00, 0.35, 0.40, 0.10, 0.30, 0.15],
    [0.35, 1.00, 0.25, 0.20, 0.20, 0.30],
    [0.40, 0.25, 1.00, 0.05, 0.35, 0.10],
    [0.10, 0.20, 0.05, 1.00, 0.15, 0.25],
    [0.30, 0.20, 0.35, 0.15, 1.00, 0.20],
    [0.15, 0.30, 0.10, 0.25, 0.20, 1.00],
])
cov = np.outer(vols, vols) * corr

n_days = 1500
rets = rng.multivariate_normal(true_mean, cov, size=n_days)
sample_mean = rets.mean(axis=0)
sample_cov = np.cov(rets.T)
r_f = 0.02

def neg_sharpe(w, mean, cov, rf):
    ret = w @ mean * 252
    vol = np.sqrt(w @ cov @ w) * np.sqrt(252)
    return -(ret - rf) / vol

w0 = np.full(n_assets, 1 / n_assets)
cons = {'type': 'eq', 'fun': lambda w: w.sum() - 1}
bounds = [(0, 1) for _ in range(n_assets)]
res = minimize(neg_sharpe, w0, args=(sample_mean, sample_cov, r_f), method='SLSQP',
                bounds=bounds, constraints=cons)
w_target = res.x
ret_target = w_target @ sample_mean * 252
vol_target = np.sqrt(w_target @ sample_cov @ w_target) * np.sqrt(252)
sharpe_target = (ret_target - r_f) / vol_target

print("TARGET (SciPy SLSQP, long-only, fully invested):")
print("  weights:", np.round(w_target, 4))
print(f"  annualized return={ret_target:.4f}  vol={vol_target:.4f}  Sharpe={sharpe_target:.4f}")

TARGET (SciPy SLSQP, long-only, fully invested):
  weights: [0.5767 0.     0.2958 0.     0.1275 0.    ]
  annualized return=0.2144  vol=0.1595  Sharpe=1.2192


The PyTorch version: raw, unconstrained scores are passed through a softmax to produce valid portfolio weights, and the optimizer directly maximizes the (negative) Sharpe ratio by gradient descent, with no explicit constraint-handling code needed since the softmax makes the constraints structurally impossible to violate.

In [ ]:
mean_t = torch.tensor(sample_mean, dtype=torch.float32)
cov_t = torch.tensor(sample_cov, dtype=torch.float32)
rf_t = torch.tensor(r_f, dtype=torch.float32)

raw_scores = torch.zeros(n_assets, requires_grad=True)
optimizer = optim.Adam([raw_scores], lr=0.05)

for step in range(500):
    optimizer.zero_grad()
    w = torch.softmax(raw_scores, dim=0)          # softmax guarantees w >= 0 and w.sum() == 1
    port_ret = (w @ mean_t) * 252
    port_vol = torch.sqrt(w @ cov_t @ w) * (252 ** 0.5)
    neg_sharpe_t = -(port_ret - rf_t) / port_vol
    neg_sharpe_t.backward()
    optimizer.step()
    if step % 100 == 0:
        print(f"step {step}: Sharpe = {-neg_sharpe_t.item():.4f}")

with torch.no_grad():
    w_final = torch.softmax(raw_scores, dim=0).numpy()
    ret_final = w_final @ sample_mean * 252
    vol_final = np.sqrt(w_final @ sample_cov @ w_final) * np.sqrt(252)
    sharpe_final = (ret_final - r_f) / vol_final

print("\nGRADIENT DESCENT RESULT (should closely match the SciPy target above):")
print("  weights:", np.round(w_final, 4))
print(f"  annualized return={ret_final:.4f}  vol={vol_final:.4f}  Sharpe={sharpe_final:.4f}")

### 2.5 An LSTM for Volatility Forecasting

Volatility clustering, calm periods and turbulent periods each persisting for a while, is exactly the kind of pattern a recurrent network is built to pick up: an LSTM (Long Short-Term Memory network) processes a sequence one step at a time, carrying a hidden state forward that can, in principle, learn to summarize recent volatility conditions the way a GARCH or EWMA model does explicitly, but through a much more flexible, learned update rule instead of a fixed formula.

The target this time is not a single correct answer to match exactly, since a trained neural network's precise output is path-dependent on initialization and optimization, but an accuracy bar to try to beat: the EWMA (RiskMetrics-style) forecast used earlier in this notebook series, a strong, simple, and well-established baseline for this exact task.

In [5]:
import numpy as np

rng = np.random.default_rng(55)

n_days = 3000
base_vol = 0.01
vol_state = np.zeros(n_days)
vol_state[0] = base_vol
for t in range(1, n_days):
    vol_state[t] = 0.94 * vol_state[t-1] + 0.06 * base_vol + rng.normal(0, 0.0008)
    vol_state[t] = max(vol_state[t], 0.003)

returns = rng.normal(0, 1, n_days) * vol_state

def realized_vol_forward(returns, horizon=5):
    n = len(returns)
    rv = np.full(n, np.nan)
    for t in range(n - horizon):
        rv[t] = np.std(returns[t+1:t+1+horizon])
    return rv

target_vol = realized_vol_forward(returns, horizon=5)

lam = 0.94
ewma_var = np.zeros(n_days)
ewma_var[0] = returns[0] ** 2
for t in range(1, n_days):
    ewma_var[t] = lam * ewma_var[t-1] + (1 - lam) * returns[t] ** 2
ewma_vol_forecast = np.sqrt(ewma_var)

valid = ~np.isnan(target_vol)
rmse_ewma = np.sqrt(np.mean((ewma_vol_forecast[valid] - target_vol[valid]) ** 2))
print(f"ACCURACY BAR TO BEAT: EWMA baseline RMSE (5-day forward realized volatility) = {rmse_ewma:.6f}")

ACCURACY BAR TO BEAT: EWMA baseline RMSE (5-day forward realized volatility) = 0.004142


The PyTorch version: a rolling window of the last 20 days of squared returns is fed through an `nn.LSTM` layer, and a small linear head maps the final hidden state to a volatility forecast. Squared returns, rather than raw returns, are used as the input feature since they carry the direct volatility signal the network needs.

In [ ]:
window = 20
X_seq, y_seq = [], []
for t in range(window, n_days - 5):
    X_seq.append(returns[t-window:t] ** 2)
    y_seq.append(target_vol[t])
X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

n_train = int(len(X_seq) * 0.8)
X_train_seq = torch.tensor(X_seq[:n_train], dtype=torch.float32).unsqueeze(-1)   # shape: (batch, seq_len, 1)
y_train_seq = torch.tensor(y_seq[:n_train], dtype=torch.float32)
X_test_seq = torch.tensor(X_seq[n_train:], dtype=torch.float32).unsqueeze(-1)
y_test_seq = torch.tensor(y_seq[n_train:], dtype=torch.float32)

class VolLSTM(nn.Module):
    def __init__(self, hidden_size=24):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)     # h_n: final hidden state, shape (1, batch, hidden_size)
        return torch.nn.functional.softplus(self.head(h_n.squeeze(0))).squeeze(-1)

vol_model = VolLSTM(hidden_size=24)
optimizer = optim.Adam(vol_model.parameters(), lr=3e-3)
loss_fn = nn.MSELoss()

for epoch in range(80):
    optimizer.zero_grad()
    pred = vol_model(X_train_seq)
    loss = loss_fn(pred, y_train_seq)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"epoch {epoch}: training MSE = {loss.item():.8f}")

vol_model.eval()
with torch.no_grad():
    pred_test = vol_model(X_test_seq).numpy()
rmse_lstm = np.sqrt(np.mean((pred_test - y_test_seq.numpy()) ** 2))
print(f"\nLSTM test RMSE = {rmse_lstm:.6f}  (EWMA baseline was {rmse_ewma:.6f})")
print("a well-tuned LSTM should land close to, and ideally slightly below, the EWMA baseline;")
print("meaningfully beating it consistently is a genuinely high bar, since EWMA is already a")
print("strong, well-matched model for volatility that follows this kind of persistent process")